In [ ]:
import pandas as pd
import os
from tensorflow import keras
from keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.text import Tokenizer
from keras.callbacks import EarlyStopping


Load data from file

In [44]:
base_dir = os.getcwd()
parent_dir = os.path.dirname(base_dir)
data_train_path = os.path.join(parent_dir, "data", "processed", "training.csv")
data_val_path = os.path.join(parent_dir, "data", "processed", "val.csv")
model_path = os.path.join(parent_dir, "model")

df_train = pd.read_csv(data_train_path)
df_val = pd.read_csv(data_val_path)

X_train = df_train['text']
y_train = df_train['label']
X_val = df_val['text']
y_val = df_val['label']

In [45]:
vocab_size = 10000
embedding_dim = 64
max_length = 50
num_classes = 3

In [ ]:
#Tokenize and vectorize
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)

X_train_seq = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_val_seq   = pad_sequences(X_val_seq, maxlen=max_length, padding='post')

In [ ]:
model = keras.Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    Dropout(0.5), # Adding dropout to avoid overfitting
    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy', # using labels = [0, 1, 2]
    optimizer='adam', 
    metrics=['accuracy'])

In [49]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [51]:
model.fit(
    X_train_seq, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_data=(X_val_seq, y_val),
    callbacks=[early_stop])

Epoch 1/10
2250/2250 [==============================] - 113s 50ms/step - loss: 0.4239 - accuracy: 0.8316 - val_loss: 0.2545 - val_accuracy: 0.9199
Epoch 2/10
2250/2250 [==============================] - 94s 42ms/step - loss: 0.3519 - accuracy: 0.8609 - val_loss: 0.2306 - val_accuracy: 0.9219
Epoch 3/10
2250/2250 [==============================] - 84s 37ms/step - loss: 0.3109 - accuracy: 0.8772 - val_loss: 0.2181 - val_accuracy: 0.9219
Epoch 4/10
2250/2250 [==============================] - 85s 38ms/step - loss: 0.2813 - accuracy: 0.8882 - val_loss: 0.2270 - val_accuracy: 0.9249
Epoch 5/10
2250/2250 [==============================] - 85s 38ms/step - loss: 0.2582 - accuracy: 0.8985 - val_loss: 0.2031 - val_accuracy: 0.9459
Epoch 6/10
2250/2250 [==============================] - 84s 37ms/step - loss: 0.2403 - accuracy: 0.9048 - val_loss: 0.1904 - val_accuracy: 0.9429
Epoch 7/10
2250/2250 [==============================] - 88s 39ms/step - loss: 0.2217 - accuracy: 0.9124 - val_loss: 0.2045 

In [55]:
import joblib

joblib.dump(model, os.path.join(model_path, "model_lstm.pkl"))

INFO:tensorflow:Assets written to: ram://ccbfa795-b9c7-4ee0-bc0d-ba4b4e01f3a7/assets


['d:\\DOCUMENT\\School\\NLP\\twitter_sentiment_classification\\model\\model_lstm.pkl']